In [1]:
import tqdm
import time
import datetime
import numpy as np
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

# to jest z 10.2025

In [ ]:
loops14_16 = pd.read_csv("data/new_time/hrs14-16_NNv1_time_matrix_loops.tsv", sep='\t')

motifs14_16 = pd.read_csv("data/new_time/hrs14-16_NNv1_time_matrix_motifs.tsv", sep='\t')

In [ ]:
loops_df = pd.read_csv("data/new_time/hrs14-16_NNv1_time_matrix_loops.tsv", sep='\t')

#motifs_df = pd.read_csv("data/new_time/hrs14-16_NNv1_time_matrix_motifs.tsv", sep='\t')

In [ ]:
loops_df.iloc[4,:].unique()

array(['L411', np.int64(0), np.int64(1)], dtype=object)

In [ ]:
empty_rows = loops_df.index[loops_df.isna().all(axis=1)]
print("Empty loops_df rows:", len(empty_rows))

Empty loops_df rows: 0


### Przygotowuję trójwymiary tensor [pętle, motywy, profile aktywności]`(417,361,2)`

Motywów jest **361** (nie 360). Jak się wyrzuci wartości NaN to może się zrobić mniej, dlatego lepiej uważać i wyrzucać kolumny (komórki) zamiast wierszy (motywów). Do tego służy opcja `axis=1` w metodzie `dropna()`.

EDA pomysły:
    - jaki jest rozkład "popularności pętli"? Tzn. w danym oknie, każda pętla występuje w jakimś % komórek... można zrobić z tego histogram

In [3]:
# This order (to_numeric(), then dropna()) deletes loop/motif names as well as true missing values.

# nie wiem czy to jest dobre z punktu widzenia czytelności kodu.

loops14_16 = loops14_16.apply(pd.to_numeric, errors='coerce')
motifs14_16 = motifs14_16.apply(pd.to_numeric, errors='coerce')

for (name, df) in [('loops14_16', loops14_16), ('motifs14_16', motifs14_16)]:
    
    print(f"\nNumber of NaN values in table {name}:")
    print(df.isna().sum().sum())

    df.dropna(axis=1, inplace=True)

    print(f"\nNaN values in {name} omitted. Verifying cleanup:")
    print(df.isna().sum().sum())


Number of NaN values in table loops14_16:
417

NaN values in loops14_16 omitted. Verifying cleanup:
0

Number of NaN values in table motifs14_16:
365

NaN values in motifs14_16 omitted. Verifying cleanup:
0


#### Testuję lokalnie, wersję dla podzbioru komórek

In [4]:
dummy_idx = 1001

loops_dummy = loops14_16.iloc[:,:dummy_idx]
motifs_dummy = motifs14_16.iloc[:,:dummy_idx]

In [5]:
activity_profiles = ["1-1", "other"]

def process_window(window: str)-> defaultdict:
    """ Compute a 3D dictionary of float values derived from chromVAR z-scores averaged over cell subpopulations
        (with structure: data[loop][motif][profile]).

    Args:
        window (str): timepoint to process (eg. 'hrs06-08')

    Returns:
        data_3D: a defaultdict of float values (average chromVAR scores for motifs in cell populations having a distinct loop activity pattern)

    """
    activity_profiles = ["1-1", "other"]

    loops_path = f"data/new_time/hrs{window}_NNv1_time_matrix_loops.tsv"
    motifs_path = f"data/new_time/hrs{window}_NNv1_time_matrix_motifs.tsv"

    loops_df = pd.read_csv(loops_path, sep='\t')
    motifs_df = pd.read_csv(motifs_path, sep='\t')

    # Preliminary processing files
    # This order (to_numeric(), then dropna()) deletes loop/motif names as well as true missing values.
    for df in [loops_df, motifs_df]:
        df = df.apply(pd.to_numeric, errors='coerce')
        df.dropna(axis=1, inplace=True)

    # Real business
    data_3D = defaultdict(lambda: defaultdict(dict))

    for loop_id in loops_df.index:

        # Identify subpopulations based on loop presence
        loop_values = loops_df.loc[loop_id]
        cells_11 = loop_values[loop_values == 11].index
        cells_other = loop_values.index.difference(cells_11)

        for motif_id in motifs_dummy.index:
            motif_values = motifs_dummy.loc[motif_id]
            # Compute mean z-score within each subpopulation
            mean_11 = motif_values.loc[cells_11].mean() if len(cells_11) > 0 else np.nan
            mean_other = motif_values.loc[cells_other].mean() if len(cells_other) > 0 else np.nan

            data_3D[loop_id][motif_id]["1-1"] = mean_11
            data_3D[loop_id][motif_id]["other"] = mean_other

    return data_3D


In [17]:
def process_pom(loops_df, motifs_df)-> defaultdict:
    """ 3D pomocnicze
    """

    # Preliminary processing files
    # This order (to_numeric(), then dropna()) deletes loop/motif names as well as true missing values.
    for df in [loops_df, motifs_df]:
        df = df.apply(pd.to_numeric, errors='coerce')
        df.dropna(axis=1, inplace=True)

    # Real business
    data_3D = defaultdict(lambda: defaultdict(dict))

    for loop_id in tqdm.tqdm(loops_df.index):

        # Identify subpopulations based on loop presence
        loop_values = loops_df.loc[loop_id]
        cells_11 = loop_values[loop_values == 11].index
        cells_other = loop_values.index.difference(cells_11)

        for motif_id in motifs_dummy.index:
            motif_values = motifs_dummy.loc[motif_id]
            # Compute mean z-score within each subpopulation
            mean_11 = motif_values.loc[cells_11].mean() if len(cells_11) > 0 else np.nan
            mean_other = motif_values.loc[cells_other].mean() if len(cells_other) > 0 else np.nan

            data_3D[loop_id][motif_id]["1-1"] = mean_11
            data_3D[loop_id][motif_id]["other"] = mean_other

    return data_3D


In [6]:
def convert_2D(data_3D: defaultdict)-> pd.DataFrame:
    """ For each (loop, motif) pair: compute the difference in mean chromVAR scores between populations of cells '1-1' and 'other'. 

    Args:  
        data_3D (defaultdict): output of the process_window() function.

    Returns:
        data_diff: a DataFrame with loops (rows), motifs (columns) and '1-1' - 'other' difference (values).
    """
    num_loops, num_motifs = len(data_3D), len(data_3D[0])
    data_2D = defaultdict(lambda: defaultdict(dict))

    records = []
    for loop_id, motif_dict in data_3D.items():
        for motif_id, profile_dict in motif_dict.items():

            mean_11 = profile_dict["1-1"]
            mean_other = profile_dict["other"]
            data_2D[loop_id][motif_id] = np.subtract(mean_11, mean_other)

    data_diff = pd.DataFrame.from_dict(data_2D, orient='index')

    return data_diff

In [21]:
def process_loop(loop_id, loops_df, motifs_df):
    """Process a single loop_id — runs in a worker process."""
    result = defaultdict(dict)

    loop_values = loops_df.loc[loop_id]
    cells_11 = loop_values[loop_values == 11].index
    cells_other = loop_values.index.difference(cells_11)

    for motif_id in motifs_df.index:
        motif_values = motifs_df.loc[motif_id]
        mean_11 = motif_values.loc[cells_11].mean() if len(cells_11) > 0 else np.nan
        mean_other = motif_values.loc[cells_other].mean() if len(cells_other) > 0 else np.nan
        result[motif_id]["1-1"] = mean_11
        result[motif_id]["other"] = mean_other

    return loop_id, result

def load_window(window: str):
    """ Load data and drop missing values in a way that

    Args:
        window (str): timepoint to process (eg. 'hrs06-08')

    Returns:
        
    """
    activity_profiles = ["1-1", "other"]

    loops_path = f"data/new_time/hrs{window}_NNv1_time_matrix_loops.tsv"
    motifs_path = f"data/new_time/hrs{window}_NNv1_time_matrix_motifs.tsv"

    print(f"{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\t\t Loading loops...\n")
    loops_df = pd.read_csv(loops_path, sep='\t')
    print(f"{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\t\t Loading motifs...\n")
    motifs_df = pd.read_csv(motifs_path, sep='\t')

    # Preliminary processing files
    # This order (to_numeric(), then dropna()) deletes loop/motif names as well as true missing values.
    print(f"{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\t\t Converting loops to numeric and dropping NaN values...\n")
    loops_df.apply(pd.to_numeric, errors='coerce')
    loops_df.dropna(axis=1, inplace=True)

    print(f"{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\t\t Converting motifs to numeric and dropping NaN values...\n")
    motifs_df = motifs_df.apply(pd.to_numeric, errors='coerce')
    motifs_df.dropna(axis=1, inplace=True)
    
    common = list(set(loops_df.columns) & set(motifs_df.columns))
    loops_df = loops_df[common]
    motifs_df = motifs_df[common]
    
    print(f"{datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\t\t Done\n")
    return loops_df, motifs_df

def process_window_parallel(window: str, n_workers=6)-> defaultdict:
    """Parallel computation of data_3D using multiple processes."""
    loops_df, motifs_df = loops_dummy, motifs_dummy#load_window(window)
    

    data_3D = defaultdict(lambda: defaultdict(dict))
    print(f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}\t\tReal business begins\n")

    # Submit tasks
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(process_loop, loop_id, loops_df, motifs_df): loop_id for loop_id in loops_df.index}

        for fut in tqdm.tqdm(as_completed(futures), total=len(futures)):
            loop_id, result = fut.result()
            data_3D[loop_id] = result

    return data_3D

In [22]:
data_diff = convert_2D(process_window_parallel('_', n_workers=8))

2025-11-11 00:02:20		Real business begins



100%|██████████| 417/417 [00:25<00:00, 16.12it/s]


In [140]:
activity_profiles = ["1-1", "other"]

# We'll use a nested dict structure: data[loop][motif][profile]
data_3D = defaultdict(lambda: defaultdict(dict))

for loop_id in loops_dummy.index:
    # Identify subpopulations based on loop presence
    loop_values = loops_dummy.loc[loop_id]
    cells_11 = loop_values[loop_values == 11].index
    cells_other = loop_values.index.difference(cells_11)

    for motif_id in motifs_dummy.index:
        motif_values = motifs_dummy.loc[motif_id]
        # Compute mean z-score within each subpopulation
        mean_11 = motif_values.loc[cells_11].mean() if len(cells_11) > 0 else np.nan
        mean_other = motif_values.loc[cells_other].mean() if len(cells_other) > 0 else np.nan

        data_3D[loop_id][motif_id]["1-1"] = mean_11
        data_3D[loop_id][motif_id]["other"] = mean_other



In [117]:
# --- Optional: convert to a more convenient structure (for analysis) ---
# For example, a pandas DataFrame with MultiIndex
records = []
for loop_id, motif_dict in data_3D.items():
    for motif_id, profile_dict in motif_dict.items():
        records.append({
            "loop": loop_id,
            "motif": motif_id,
            "mean_score_1-1": profile_dict["1-1"],
            "mean_score_other": profile_dict["other"]
        })

loop_motif_activity = pd.DataFrame(records)
print(loop_motif_activity.head())

   loop  motif  mean_score_1-1  mean_score_other
0     0      0       -0.369463          0.048548
1     0      1        1.551062          0.595379
2     0      2        3.704142          1.205620
3     0      3        3.974245          1.205163
4     0      4        2.104314          0.660553
